This document aims to accomplish the following, for the yearly sample only (devices seen on 350+ days of the year, [2ai], with their cleaned dispensary visits from [4bi]):

(a) Flag possible retail workers: devices that keep coming back to the same store (license) on weekdays, stay for a long time, and come and go at regular times

(i) One row per device, license, and quarter, with every criterion measured. Label as ([7ai])

(ii) The number and share of devices flagged, by quarter and year, for k = 25, 33, 50, 66, 75, and 90 (the share of entrance or exit times inside the best half hour, or, as an alternative, the best hour), under the main rule (4+ weekdays in 9+ weeks) and three looser ones: 8+ weeks, 6+ weeks, and 3+ weekdays in 9+ weeks. Label as ([7aii])

(b) Flag possible delivery or wholesale workers: devices making frequent, recurring visits across several stores, and staying long

(i) One row per device and quarter, with every criterion measured. Label as ([7bi])

(ii) The number and share of devices flagged, by quarter and year, under the main rule (9+ weeks) and looser ones (8+ and 6+ weeks). Label as ([7bii])

(c) Flag high-frequency devices: more than 31 dispensary visits in a calendar month

(i) One row per device and month with a dispensary visit. Label as ([7ci])

(ii) The number and share of devices flagged, by month and year. Label as ([7cii])

(d) One row per sample device and quarter with all three flags (main rules, k = 75 for the retail flag), to carry forward. Label as ([7d]), with the counts, shares, and overlaps as ([7d'])

(e) Visit summary statistics before and after excluding the flagged quarters (retail, delivery) and months (high frequency), each flag on its own and all of them

(i) frequency: the share of devices with a visit, and among them visits, visits per 30 observed days, and stores. Label as ([7ei])

(ii) time: start times, days of the week, and dwell. Label as ([7eii]), with start hours as ([7eii'])

(iii) region: the share of visits to stores in the home county, city, ZIP, and CBG, and the share of visits by store county. Label as ([7eiii]), with store counties as ([7eiii'])

Notes:
- Weeks run Monday to Sunday, weekdays are Monday-Friday, and months and quarters are local calendar ones (not the UTC month folder in [4bi], which differs for 1,994 visits). A week counts toward the quarter its visits are in (a week straddling two quarters is split between them)
- "Same store" is the same WSLCB license, wherever it was. A visit to a storefront with two licenses operating that month counts for both. A device can be a retail worker for more than one store, and is flagged for the quarter if it qualifies at any of them
- A visit "lasts at least an hour" when its minimum dwell is 60 minutes or more. Visits Veraset cut at UTC midnight were put back together in Step 4 ([4bi])
- Flags are 1/0 and belong to the period they were measured in: a device is a retail or delivery worker in a quarter, and high frequency in a month. Excluding a flag drops only the visits in the flagged quarters (or months); the device's other quarters stay in. Counts for a year add up the flagged device-quarters (or device-months)
- Charts also get saved to "Task 2/Charts and Figures" as .html files


In [1]:
### IMPORTS AND PRELIMINARIES
## MEMORY STUFF
# Stops polars from holding onto freed memory, and forces it to be returned to the OS
import os
os.environ['_RJEM_MALLOC_CONF'] = "background_thread:true,dirty_decay_ms:0,muzzy_decay_ms:0"

## WORKHORSE LIBRARY
# Enables data streaming of large files
import polars as pl

## CHARTS (same as week 6)
import numpy as np
import plotly.express as px

# Cleans up some cell outputs
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

YEARS = ["2019", "2020", "2021"]
STOREFRONT = ["street_address_norm", "city_norm"]
CHART_FOLDER = "Task 2/Charts and Figures"
os.makedirs(CHART_FOLDER, exist_ok = True)

def show(fig, name):
    fig.show()
    fig.write_html(f"{CHART_FOLDER}/{name}.html", include_plotlyjs = "cdn") # so they can be opened outside the notebook too

<jemalloc>: option background_thread currently supports pthread only


In [2]:
### LOADING
## Every yearly sample device ([2ai], with its local calendar year window and observed days from Step 4), its home ([4ai]), and its dispensary visits ([4bi])
windows = pl.read_parquet("processed-data/device_windows_yearly.parquet")
crosswalk = pl.read_parquet("processed-data/4ai_device_home_geography_yearly.parquet")

# local_timestamp is local time written as if it were UTC, so plain UTC date parts are local
visits = (pl.read_parquet("processed-data/4bi_dispensary_visits_yearly.parquet")
    .with_columns(
        pl.from_epoch("local_timestamp").dt.weekday().alias("day_of_week"), # 1 = Monday .. 7 = Sunday
        pl.from_epoch("local_timestamp").dt.hour().alias("hour"),
        pl.from_epoch("local_timestamp").dt.strftime("%Y%m").cast(pl.Int32).alias("month"), # the local calendar month (the one in [4bi] is Veraset's UTC folder)
    )
    .with_columns(
        (pl.col("year").cast(pl.String) + "Q" + ((pl.col("month") % 100 - 1) // 3 + 1).cast(pl.String)).alias("quarter"),
        (pl.col("minimum_dwell") >= 60).alias("hour_plus"),
        (pl.col("local_timestamp") % 86400).alias("start_second"), # second of the (local) day the visit starts
        ((pl.col("local_timestamp") + (pl.col("minimum_dwell") * 60).cast(pl.Int64)) % 86400).alias("end_second"), # and ends (start + dwell)
    )
    .with_columns((pl.col("day") - pl.col("day_of_week") + 1).alias("week_monday")) # day number of the Monday starting the visit's week
)

print("sample device-years:", windows.shape[0], "| with a home:", crosswalk.shape[0], "| dispensary visits:", visits.shape[0],
      "| device-years with a visit:", visits.select(["year", "caid_hash"]).unique().shape[0])
windows.group_by("year").agg(pl.len().alias("devices"), pl.col("observed_days").mean().round(1).alias("mean_observed_days")).sort("year")

sample device-years: 87962 | with a home: 87962 | dispensary visits: 116273 | device-years with a visit: 20608


year,devices,mean_observed_days
i32,u32,f64
2019,23552,356.7
2020,38224,358.6
2021,26186,357.9


In [3]:
### HELPER
## Best half hour: the largest share of a set of times of day that fit in one window of width seconds (closed at both ends), sliding the window
## around the clock (so 11:50 pm and 12:10 am are 20 minutes apart). Returns (share, start of the best window, "HH:MM")
WINDOW = 1800 # seconds, half an hour (the main version)
WINDOW_LONG = 3600 # an hour, the alternative in [7aii]

def best_window(seconds, width = WINDOW):
    times = np.sort(np.asarray(seconds, dtype = np.int64))
    n = len(times)
    if n == 0:
        return (None, None)
    wrapped = np.concatenate([times, times + 86400])
    counts = np.searchsorted(wrapped, times + width, side = "right") - np.arange(n) # times from each one to width after it
    best = int(counts.argmax())
    return (counts[best] / n, f"{times[best] // 3600:02d}:{(times[best] % 3600) // 60:02d}")

print(best_window([0, 600, 1800, 3601, 86000]), best_window([86000, 100, 40000, 40001]), best_window([0, 600, 1800, 3601, 86000], WINDOW_LONG)) # (0.6, '00:00'), (0.5, '11:06'): 86000 and 100 are also 0.5, but 11:06 comes first, and (0.8, '23:53') with an hour

## Counts and shares of flagged device-periods (device-quarters, or device-months), out of every sample device in the period and out of the ones with a
## dispensary visit then. A year (and "all") adds up its device-periods, so a device flagged in two quarters counts twice and one flagged in one quarter
## counts once, against four device-quarters
def flag_counts(flags, period, stages):
    # flags: one row per (period, caid_hash) with a "year" column and a 1/0 column per stage
    devices = windows.group_by("year").len().rename({"len": "device_periods_in_sample"})
    visitors = visits.group_by(period).agg(pl.col("caid_hash").n_unique().alias("device_periods_visiting"))
    by_period = (visits.select([period, "year"]).unique()
        .join(flags.group_by(period).agg([pl.col(s).sum() for s in stages]), on = period, how = "left")
        .with_columns(pl.col(stages).fill_null(0))
        .join(devices, on = "year").join(visitors, on = period)
        .with_columns(pl.col(period).cast(pl.String).alias("period"))
        .sort("period"))
    columns = stages + ["device_periods_in_sample", "device_periods_visiting"]
    by_year = by_period.group_by("year").agg([pl.col(c).sum() for c in columns]).with_columns(pl.col("year").cast(pl.String).alias("period")).sort("period")
    pooled = by_period.select([pl.lit("all").alias("period")] + [pl.col(c).sum() for c in columns])
    columns = ["period", "device_periods_in_sample", "device_periods_visiting"] + stages
    table = pl.concat([by_period.select(columns), by_year.select(columns), pooled.select(columns)], how = "vertical_relaxed")
    final = stages[-1]
    return table.with_columns(
        (pl.col(final) / pl.col("device_periods_in_sample")).alias("share_of_sample"),
        (pl.col(final) / pl.col("device_periods_visiting")).alias("share_of_visitors"),
    )

(np.float64(0.6), '00:00') (np.float64(0.5), '11:06') (np.float64(0.8), '23:53')


In [4]:
### PART A, SUBPART I
## Output: [7ai] one row per device, license, and quarter with a dispensary visit to that license, and every retail worker criterion
##         Columns are "year", "quarter", "caid_hash", "license", "visits" (to that license, that quarter), "weekdays_visited" (distinct local Monday-Friday
##         days), "qualifying_weeks" (weeks with visits to the license on 4+ distinct weekdays), "qualifying_weeks_3days" (the same with 3+ weekdays,
##         for the looser rule in [7aii]), "share_hour_plus" (share of the visits to the license with
##         a minimum dwell of 60+ minutes), "start_window_share" / "start_window" (the most entrance times inside any one half hour, and where that half
##         hour starts), "end_window_share" / "end_window" (the same for exit times, start + dwell), "regular_share" (the larger of the two), the same
##         four with the best hour instead ("start_window_share_1h", "start_window_1h", "end_window_share_1h", "end_window_1h", "regular_share_1h"), then
##         "meets_weeks" (9+ qualifying weeks), "meets_dwell" (75%+ of visits an hour or longer), and the flag for each k, "retail_k25" ... "retail_k90"
##         (meets_weeks, meets_dwell, and regular_share >= k; the main half-hour version)
## Name: 7ai_retail_worker_criteria.parquet
MIN_WEEKDAYS = 4 # distinct weekdays at the store in a week
MIN_WEEKS = 9 # weeks like that in the quarter
MIN_HOUR_PLUS = 0.75 # share of the visits lasting an hour or more
K_VALUES = [25, 33, 50, 66, 75, 90] # percent of entrance (or exit) times inside the best half hour
K_MAIN = 75 # the one carried forward

# (1) One row per visit and license (a storefront with two licenses operating that month counts for both)
license_visits = (visits
    .explode("operating_licenses")
    .rename({"operating_licenses": "license"})
    .drop_nulls("license")
)

# (2) Weeks with visits to the license on 4+ distinct weekdays
weeks = (license_visits
    .filter(pl.col("day_of_week") <= 5)
    .group_by(["year", "quarter", "caid_hash", "license", "week_monday"])
    .agg(pl.col("day").n_unique().alias("weekdays"))
    .group_by(["year", "quarter", "caid_hash", "license"])
    .agg(
        pl.col("weekdays").sum().alias("weekdays_visited"),
        (pl.col("weekdays") >= MIN_WEEKDAYS).sum().alias("qualifying_weeks"),
        (pl.col("weekdays") >= 3).sum().alias("qualifying_weeks_3days"), # looser version, see [7aii]
    )
)

# (3) Dwell, and how regular the entrance and exit times are
retail = (license_visits
    .group_by(["year", "quarter", "caid_hash", "license"])
    .agg(pl.len().alias("visits"), pl.col("hour_plus").mean().alias("share_hour_plus"), pl.col("start_second"), pl.col("end_second"))
    .join(weeks, on = ["year", "quarter", "caid_hash", "license"], how = "left")
    .with_columns(pl.col(["weekdays_visited", "qualifying_weeks", "qualifying_weeks_3days"]).fill_null(0))
)
starts = [best_window(s) for s in retail["start_second"].to_list()]
ends = [best_window(s) for s in retail["end_second"].to_list()]
starts_1h = [best_window(s, WINDOW_LONG) for s in retail["start_second"].to_list()]
ends_1h = [best_window(s, WINDOW_LONG) for s in retail["end_second"].to_list()]
retail = (retail
    .drop(["start_second", "end_second"])
    .with_columns(
        pl.Series("start_window_share", [s[0] for s in starts]), pl.Series("start_window", [s[1] for s in starts]),
        pl.Series("end_window_share", [e[0] for e in ends]), pl.Series("end_window", [e[1] for e in ends]),
        pl.Series("start_window_share_1h", [s[0] for s in starts_1h]), pl.Series("start_window_1h", [s[1] for s in starts_1h]),
        pl.Series("end_window_share_1h", [e[0] for e in ends_1h]), pl.Series("end_window_1h", [e[1] for e in ends_1h]),
    )
    .with_columns(
        pl.max_horizontal("start_window_share", "end_window_share").alias("regular_share"),
        pl.max_horizontal("start_window_share_1h", "end_window_share_1h").alias("regular_share_1h"),
        (pl.col("qualifying_weeks") >= MIN_WEEKS).alias("meets_weeks"),
        (pl.col("share_hour_plus") >= MIN_HOUR_PLUS).alias("meets_dwell"),
    )
    .with_columns([(pl.col("meets_weeks") & pl.col("meets_dwell") & (pl.col("regular_share") >= K / 100)).cast(pl.Int8).alias(f"retail_k{K}")
                   for K in K_VALUES])
    .sort(["quarter", "caid_hash", "license"])
)
retail.write_parquet("processed-data/7ai_retail_worker_criteria.parquet")

print("device-license-quarters:", retail.shape[0], "| 9+ weeks with 4+ weekdays:", retail["meets_weeks"].sum(),
      "| and 75%+ of visits an hour or more:", (retail["meets_weeks"] & retail["meets_dwell"]).sum())
retail.filter("meets_weeks").sort("visits", descending = True).head(10)

device-license-quarters: 49433 | 9+ weeks with 4+ weekdays: 59 | and 75%+ of visits an hour or more: 35


year,quarter,caid_hash,license,visits,share_hour_plus,weekdays_visited,qualifying_weeks,qualifying_weeks_3days,start_window_share,start_window,end_window_share,end_window,start_window_share_1h,start_window_1h,end_window_share_1h,end_window_1h,regular_share,regular_share_1h,meets_weeks,meets_dwell,retail_k25,retail_k33,retail_k50,retail_k66,retail_k75,retail_k90
i32,str,u64,i64,u32,f64,u32,u32,u32,f64,str,f64,str,f64,str,f64,str,f64,f64,bool,bool,i8,i8,i8,i8,i8,i8
2021,"""2021Q4""",18326306263360024832,353928,284,0.126761,49,9,10,0.123239,"""09:33""",0.130282,"""10:10""",0.169014,"""13:45""",0.140845,"""09:44""",0.130282,0.169014,true,false,0,0,0,0,0,0
2021,"""2021Q3""",18326306263360024832,353928,251,0.223108,61,12,13,0.135458,"""23:55""",0.139442,"""00:13""",0.179283,"""23:55""",0.191235,"""00:10""",0.139442,0.191235,true,false,0,0,0,0,0,0
2019,"""2019Q1""",15361332944443340227,412370,137,0.437956,59,13,13,0.372263,"""06:01""",0.240876,"""07:20""",0.408759,"""05:49""",0.357664,"""06:51""",0.372263,0.408759,true,false,0,0,0,0,0,0
2021,"""2021Q1""",410815880433223875,415238,137,0.649635,56,11,13,0.094891,"""17:04""",0.094891,"""16:30""",0.153285,"""17:01""",0.138686,"""16:28""",0.094891,0.153285,true,false,0,0,0,0,0,0
2020,"""2020Q4""",16591409552601883505,412921,133,0.390977,65,13,13,0.466165,"""19:51""",0.43609,"""20:34""",0.466165,"""19:51""",0.466165,"""20:32""",0.466165,0.466165,true,false,0,0,0,0,0,0
2021,"""2021Q2""",13439850626153095776,417469,132,0.613636,51,10,13,0.416667,"""07:21""",0.166667,"""11:37""",0.439394,"""07:02""",0.280303,"""11:08""",0.416667,0.439394,true,false,0,0,0,0,0,0
2021,"""2021Q2""",13439850626153095776,439653,131,0.610687,50,10,13,0.412214,"""07:21""",0.167939,"""11:37""",0.435115,"""07:02""",0.274809,"""11:01""",0.412214,0.435115,true,false,0,0,0,0,0,0
2021,"""2021Q3""",13439850626153095776,417469,131,0.549618,56,11,13,0.29771,"""07:15""",0.152672,"""11:05""",0.389313,"""07:20""",0.221374,"""10:33""",0.29771,0.389313,true,false,0,0,0,0,0,0
2021,"""2021Q3""",13439850626153095776,439653,131,0.549618,56,11,13,0.29771,"""07:15""",0.152672,"""11:05""",0.389313,"""07:20""",0.221374,"""10:33""",0.29771,0.389313,true,false,0,0,0,0,0,0


In [5]:
### PART A, SUBPART II
## Output: [7aii] how many device-quarters get the retail worker flag, by quarter, year (its device-quarters added up), and all years, for each rule and k
##         Rules are "9 weeks, 4+ weekdays" (the main one, as in [7ai]), and the looser "8 weeks, 4+ weekdays", "6 weeks, 4+ weekdays", and
##         "9 weeks, 3+ weekdays" (the dwell and regular-times criteria stay the same). "window" is the regular-times window: "30 min" (the main one) or
##         "60 min" (k% of entrance or exit times inside the best hour)
##         Columns are "window", "rule", "k", "period", "device_periods_in_sample" (every [2ai] device that year, once per quarter), "device_periods_visiting" (with
##         a dispensary visit that quarter), "meets_weeks", "meets_weeks_and_dwell" (device-quarters passing the first criteria, at some store), "flagged"
##         (and regular times), "share_of_sample", "share_of_visitors"
## Name: 7aii_retail_worker_counts.parquet
RETAIL_RULES = [ # (name, weeks needed, which weekly count)
    ("9 weeks, 4+ weekdays", 9, "qualifying_weeks"),
    ("8 weeks, 4+ weekdays", 8, "qualifying_weeks"),
    ("6 weeks, 4+ weekdays", 6, "qualifying_weeks"),
    ("9 weeks, 3+ weekdays", 9, "qualifying_weeks_3days"),
]
MAIN_RULE = RETAIL_RULES[0][0]

WINDOWS = [("30 min", "regular_share"), ("60 min", "regular_share_1h")]

results = []
for window, share_column in WINDOWS:
    for rule, weeks_needed, week_column in RETAIL_RULES:
        rule_rows = retail.with_columns((pl.col(week_column) >= weeks_needed).alias("rule_weeks"))
        for K in K_VALUES:
            device_quarters = (rule_rows
                .group_by(["year", "quarter", "caid_hash"])
                .agg(
                    pl.col("rule_weeks").max().cast(pl.Int8).alias("meets_weeks"),
                    (pl.col("rule_weeks") & pl.col("meets_dwell")).max().cast(pl.Int8).alias("meets_weeks_and_dwell"),
                    (pl.col("rule_weeks") & pl.col("meets_dwell") & (pl.col(share_column) >= K / 100)).max().cast(pl.Int8).alias("flagged"),
                )
            )
            results.append(flag_counts(device_quarters, "quarter", ["meets_weeks", "meets_weeks_and_dwell", "flagged"])
                           .with_columns(pl.lit(window).alias("window"), pl.lit(rule).alias("rule"), pl.lit(K).alias("k")))

retail_counts = pl.concat(results).select(["window", "rule", "k", "period", "device_periods_in_sample", "device_periods_visiting", "meets_weeks", "meets_weeks_and_dwell", "flagged",
                                           "share_of_sample", "share_of_visitors"])
retail_counts.write_parquet("processed-data/7aii_retail_worker_counts.parquet")

# (1) Device-quarters flagged (all years), by window, rule, and k
(retail_counts
    .filter(pl.col("period") == "all")
    .pivot(on = "k", index = ["window", "rule"], values = "flagged", maintain_order = True))

window,rule,25,33,50,66,75,90
str,str,i64,i64,i64,i64,i64,i64
"""30 min""","""9 weeks, 4+ weekdays""",30,29,18,8,6,1
"""30 min""","""8 weeks, 4+ weekdays""",42,41,27,14,8,2
"""30 min""","""6 weeks, 4+ weekdays""",58,55,36,20,12,3
"""30 min""","""9 weeks, 3+ weekdays""",58,56,35,19,14,5
"""60 min""","""9 weeks, 4+ weekdays""",31,31,22,11,8,1
"""60 min""","""8 weeks, 4+ weekdays""",43,43,32,19,13,2
"""60 min""","""6 weeks, 4+ weekdays""",59,59,43,26,20,4
"""60 min""","""9 weeks, 3+ weekdays""",60,59,41,25,19,6


In [6]:
# (2) The same as shares of the devices with a dispensary visit (in %), and the steps before the regular-times test
display((retail_counts
    .filter(pl.col("period") == "all")
    .with_columns((pl.col("share_of_visitors") * 100).round(3))
    .pivot(on = "k", index = ["window", "rule"], values = "share_of_visitors", maintain_order = True)))
retail_counts.filter((pl.col("period") == "all") & (pl.col("k") == K_MAIN) & (pl.col("window") == "30 min")).select(["rule", "device_periods_visiting", "meets_weeks", "meets_weeks_and_dwell", "flagged"])

window,rule,25,33,50,66,75,90
str,str,f64,f64,f64,f64,f64,f64
"""30 min""","""9 weeks, 4+ weekdays""",0.079,0.077,0.048,0.021,0.016,0.003
"""30 min""","""8 weeks, 4+ weekdays""",0.111,0.108,0.071,0.037,0.021,0.005
"""30 min""","""6 weeks, 4+ weekdays""",0.153,0.145,0.095,0.053,0.032,0.008
"""30 min""","""9 weeks, 3+ weekdays""",0.153,0.148,0.092,0.05,0.037,0.013
"""60 min""","""9 weeks, 4+ weekdays""",0.082,0.082,0.058,0.029,0.021,0.003
"""60 min""","""8 weeks, 4+ weekdays""",0.114,0.114,0.085,0.05,0.034,0.005
"""60 min""","""6 weeks, 4+ weekdays""",0.156,0.156,0.114,0.069,0.053,0.011
"""60 min""","""9 weeks, 3+ weekdays""",0.158,0.156,0.108,0.066,0.05,0.016


rule,device_periods_visiting,meets_weeks,meets_weeks_and_dwell,flagged
str,u32,i64,i64,i64
"""9 weeks, 4+ weekdays""",37869,52,31,6
"""8 weeks, 4+ weekdays""",37869,70,43,8
"""6 weeks, 4+ weekdays""",37869,93,60,12
"""9 weeks, 3+ weekdays""",37869,94,61,14


In [7]:
# (3) Year by year, k = 75, for each window and rule
(retail_counts
    .filter(pl.col("period").is_in(YEARS + ["all"]) & (pl.col("k") == K_MAIN))
    .pivot(on = "period", index = ["window", "rule"], values = "flagged", maintain_order = True))

window,rule,2019,2020,2021,all
str,str,i64,i64,i64,i64
"""30 min""","""9 weeks, 4+ weekdays""",1,4,1,6
"""30 min""","""8 weeks, 4+ weekdays""",1,6,1,8
"""30 min""","""6 weeks, 4+ weekdays""",3,6,3,12
"""30 min""","""9 weeks, 3+ weekdays""",2,6,6,14
"""60 min""","""9 weeks, 4+ weekdays""",2,5,1,8
"""60 min""","""8 weeks, 4+ weekdays""",2,8,3,13
"""60 min""","""6 weeks, 4+ weekdays""",4,11,5,20
"""60 min""","""9 weeks, 3+ weekdays""",3,9,7,19


In [8]:
# (4) The main rule quarter by quarter, k = 75, with the steps along the way
retail_counts.filter((pl.col("window") == "30 min") & (pl.col("rule") == MAIN_RULE) & (pl.col("k") == K_MAIN)).with_columns(pl.col(["share_of_sample", "share_of_visitors"]).round(5))

window,rule,k,period,device_periods_in_sample,device_periods_visiting,meets_weeks,meets_weeks_and_dwell,flagged,share_of_sample,share_of_visitors
str,str,i32,str,u32,u32,i64,i64,i64,f64,f64
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2019Q1""",23552,2093,4,2,0,0.0,0.0
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2019Q2""",23552,2337,5,2,0,0.0,0.0
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2019Q3""",23552,2434,3,1,1,0.00004,0.00041
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2019Q4""",23552,2148,1,0,0,0.0,0.0
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2020Q1""",38224,3394,5,4,2,0.00005,0.00059
…,…,…,…,…,…,…,…,…,…,…
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2021Q4""",26186,3618,8,5,1,0.00004,0.00028
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2019""",94208,9012,13,5,1,0.00001,0.00011
"""30 min""","""9 weeks, 4+ weekdays""",75,"""2020""",152896,14530,19,15,4,0.00003,0.00028


In [9]:
# (5) Charts: share of sample device-quarters flagged as possible retail workers, by k and rule (all years), one for the best half hour and one
#     for the best hour
for window, title_window, name in [("30 min", "Half Hour", "retail_workers_by_k"), ("60 min", "Hour", "retail_workers_by_k_1h_window")]:
    plot_df = (retail_counts
        .filter((pl.col("period") == "all") & (pl.col("window") == window))
        .with_columns((pl.col("share_of_sample") * 100).alias("share_pct"), pl.col("k").cast(pl.String) + "%")
    )

    fig1 = px.bar(
        plot_df.to_pandas(),
        x="k",
        y="share_pct",
        color="rule",
        barmode="group",
        text="flagged",
        labels={
            "k": f"Share of entrance or exit times in the best {title_window.lower()} (k)",
            "share_pct": "Share of sample device-quarters flagged (%)",
            "rule": "Weeks rule",
        },
        title=f"Possible Retail Workers by Regular-Times Threshold k and Weeks Rule (Best {title_window}), Yearly Sample 2019-2021",
    )

    show(fig1, name)

In [10]:
### PART B, SUBPART I
## Output: [7bi] one row per device and quarter with a dispensary visit, and every delivery/wholesale worker criterion
##         Columns are "year", "quarter", "caid_hash", "visits", "storefronts" (distinct WSLCB storefronts that quarter), "active_weeks" (weeks with a visit),
##         "qualifying_weeks" (weeks with 3+ visits to 2+ different storefronts), "share_hour_plus" (share of all the device's dispensary visits that
##         quarter lasting 60+ minutes), "meets_weeks" (9+ qualifying weeks), "meets_dwell" (75%+ an hour or longer), "delivery_worker" (both, 1/0)
## Name: 7bi_delivery_worker_criteria.parquet
MIN_WEEKLY_VISITS = 3
MIN_WEEKLY_STOREFRONTS = 2

# (1) Weeks with 3+ visits that reach 2+ storefronts
delivery_weeks = (visits
    .group_by(["year", "quarter", "caid_hash", "week_monday"])
    .agg(pl.len().alias("week_visits"), pl.struct(STOREFRONT).n_unique().alias("week_storefronts"))
    .group_by(["year", "quarter", "caid_hash"])
    .agg(
        pl.len().alias("active_weeks"),
        ((pl.col("week_visits") >= MIN_WEEKLY_VISITS) & (pl.col("week_storefronts") >= MIN_WEEKLY_STOREFRONTS)).sum().alias("qualifying_weeks"),
    )
)

# (2) Every visit that quarter, for the dwell criterion
delivery = (visits
    .group_by(["year", "quarter", "caid_hash"])
    .agg(pl.len().alias("visits"), pl.struct(STOREFRONT).n_unique().alias("storefronts"), pl.col("hour_plus").mean().alias("share_hour_plus"))
    .join(delivery_weeks, on = ["year", "quarter", "caid_hash"], how = "left")
    .with_columns(
        (pl.col("qualifying_weeks") >= MIN_WEEKS).alias("meets_weeks"),
        (pl.col("share_hour_plus") >= MIN_HOUR_PLUS).alias("meets_dwell"),
    )
    .with_columns((pl.col("meets_weeks") & pl.col("meets_dwell")).cast(pl.Int8).alias("delivery_worker"))
    .select(["year", "quarter", "caid_hash", "visits", "storefronts", "active_weeks", "qualifying_weeks", "share_hour_plus", "meets_weeks", "meets_dwell", "delivery_worker"])
    .sort(["quarter", "caid_hash"])
)
delivery.write_parquet("processed-data/7bi_delivery_worker_criteria.parquet")

print("device-quarters with a visit:", delivery.shape[0], "| 9+ weeks with 3+ visits to 2+ storefronts:", delivery["meets_weeks"].sum(),
      "| and 75%+ of visits an hour or more:", delivery["delivery_worker"].sum())
delivery.sort("qualifying_weeks", descending = True).head(10)

device-quarters with a visit: 37869 | 9+ weeks with 3+ visits to 2+ storefronts: 14 | and 75%+ of visits an hour or more: 1


year,quarter,caid_hash,visits,storefronts,active_weeks,qualifying_weeks,share_hour_plus,meets_weeks,meets_dwell,delivery_worker
i32,str,u64,u32,u32,u32,u32,f64,bool,bool,i8
2019,"""2019Q3""",13734580318034118262,118,66,14,13,0.016949,true,false,0
2019,"""2019Q1""",13734580318034118262,132,79,12,12,0.0,true,false,0
2019,"""2019Q2""",13734580318034118262,142,85,13,12,0.007042,true,false,0
2020,"""2020Q1""",3660607356071084667,128,4,13,12,0.734375,true,false,0
2019,"""2019Q1""",14394546628435999842,66,38,13,11,0.015152,true,false,0
2019,"""2019Q2""",14394546628435999842,74,52,12,11,0.013514,true,false,0
2019,"""2019Q3""",14394546628435999842,63,44,13,11,0.0,true,false,0
2021,"""2021Q3""",17420892731461325325,60,3,14,11,0.15,true,false,0
2021,"""2021Q4""",9001391017427156517,95,55,13,11,0.063158,true,false,0


In [11]:
### PART B, SUBPART II
## Output: [7bii] how many device-quarters get the delivery/wholesale worker flag, by quarter, year (added up), and all years, for each rule
##         Rules are "9 weeks" (the main one, as in [7bi]), and the looser "8 weeks" and "6 weeks" (qualifying weeks needed in the quarter; the weekly
##         3+ visits to 2+ storefronts and the dwell criterion stay the same)
##         Columns are "rule", "period", "device_periods_in_sample", "device_periods_visiting", "meets_weeks", "flagged", "share_of_sample", "share_of_visitors"
## Name: 7bii_delivery_worker_counts.parquet
DELIVERY_RULES = [("9 weeks", 9), ("8 weeks", 8), ("6 weeks", 6)]

results = []
for rule, weeks_needed in DELIVERY_RULES:
    device_quarters = delivery.select(
        "year", "quarter", "caid_hash",
        (pl.col("qualifying_weeks") >= weeks_needed).cast(pl.Int8).alias("meets_weeks"),
        ((pl.col("qualifying_weeks") >= weeks_needed) & pl.col("meets_dwell")).cast(pl.Int8).alias("flagged"),
    )
    results.append(flag_counts(device_quarters, "quarter", ["meets_weeks", "flagged"]).with_columns(pl.lit(rule).alias("rule")))

delivery_counts = pl.concat(results).select(["rule", "period", "device_periods_in_sample", "device_periods_visiting", "meets_weeks", "flagged", "share_of_sample", "share_of_visitors"])
delivery_counts.write_parquet("processed-data/7bii_delivery_worker_counts.parquet")

# (1) Year by year and all years, for each rule
(delivery_counts
    .filter(pl.col("period").is_in(YEARS + ["all"]))
    .with_columns(pl.col(["share_of_sample", "share_of_visitors"]).round(5)))

rule,period,device_periods_in_sample,device_periods_visiting,meets_weeks,flagged,share_of_sample,share_of_visitors
str,str,u32,u32,i64,i64,f64,f64
"""9 weeks""","""2019""",94208,9012,7,0,0.0,0.0
"""9 weeks""","""2020""",152896,14530,2,1,0.00001,0.00007
"""9 weeks""","""2021""",104744,14327,5,0,0.0,0.0
"""9 weeks""","""all""",351848,37869,14,1,0.0,0.00003
"""8 weeks""","""2019""",94208,9012,7,0,0.0,0.0
…,…,…,…,…,…,…,…
"""8 weeks""","""all""",351848,37869,19,1,0.0,0.00003
"""6 weeks""","""2019""",94208,9012,8,0,0.0,0.0
"""6 weeks""","""2020""",152896,14530,10,2,0.00001,0.00014


In [12]:
# (2) The main rule quarter by quarter
delivery_counts.filter(pl.col("rule") == "9 weeks").with_columns(pl.col(["share_of_sample", "share_of_visitors"]).round(5))

rule,period,device_periods_in_sample,device_periods_visiting,meets_weeks,flagged,share_of_sample,share_of_visitors
str,str,u32,u32,i64,i64,f64,f64
"""9 weeks""","""2019Q1""",23552,2093,2,0,0.0,0.0
"""9 weeks""","""2019Q2""",23552,2337,2,0,0.0,0.0
"""9 weeks""","""2019Q3""",23552,2434,2,0,0.0,0.0
"""9 weeks""","""2019Q4""",23552,2148,1,0,0.0,0.0
"""9 weeks""","""2020Q1""",38224,3394,1,0,0.0,0.0
…,…,…,…,…,…,…,…
"""9 weeks""","""2021Q4""",26186,3618,2,0,0.0,0.0
"""9 weeks""","""2019""",94208,9012,7,0,0.0,0.0
"""9 weeks""","""2020""",152896,14530,2,1,0.00001,0.00007


In [13]:
### PART C, SUBPART I
## Output: [7ci] one row per device and (local) calendar month with a dispensary visit
##         Columns are "year", "month" (YYYYMM), "caid_hash", "visits", "storefronts", "visit_days" (distinct local days with a visit), "share_hour_plus",
##         "high_frequency" (more than 31 visits that month, 1/0)
## Name: 7ci_high_frequency_months.parquet
MAX_MONTHLY_VISITS = 31

high_frequency = (visits
    .group_by(["year", "month", "caid_hash"])
    .agg(
        pl.len().alias("visits"),
        pl.struct(STOREFRONT).n_unique().alias("storefronts"),
        pl.col("day").n_unique().alias("visit_days"),
        pl.col("hour_plus").mean().alias("share_hour_plus"),
    )
    .with_columns((pl.col("visits") > MAX_MONTHLY_VISITS).cast(pl.Int8).alias("high_frequency"))
    .sort(["month", "caid_hash"])
)
high_frequency.write_parquet("processed-data/7ci_high_frequency_months.parquet")

print("device-months with a visit:", high_frequency.shape[0], "| with more than 31 visits:", high_frequency["high_frequency"].sum())
high_frequency.filter(pl.col("high_frequency") == 1).sort("visits", descending = True).head(10)

device-months with a visit: 57382 | with more than 31 visits: 133


year,month,caid_hash,visits,storefronts,visit_days,share_hour_plus,high_frequency
i32,i32,u64,u32,u32,u32,f64,i8
2021,202110,18326306263360024832,142,1,31,0.161972,1
2021,202111,18326306263360024832,126,1,26,0.087302,1
2021,202107,18326306263360024832,94,1,31,0.297872,1
2021,202108,18326306263360024832,91,1,27,0.175824,1
2021,202107,18210963314565493349,89,2,29,0.202247,1
2021,202109,18210963314565493349,66,2,17,0.242424,1
2021,202109,18326306263360024832,66,1,28,0.181818,1
2019,201904,13734580318034118262,56,42,18,0.0,1
2021,202103,410815880433223875,56,1,27,0.803571,1


In [14]:
### PART C, SUBPART II
## Output: [7cii] how many device-months get the high-frequency flag, by month, year (its device-months added up), and all years
##         Columns are "period", "device_periods_in_sample" (every [2ai] device that year, once per month), "device_periods_visiting", "flagged",
##         "share_of_sample", "share_of_visitors"
## Name: 7cii_high_frequency_counts.parquet
high_frequency_counts = flag_counts(high_frequency.select(["year", "month", "caid_hash", pl.col("high_frequency").alias("flagged")]), "month", ["flagged"])
high_frequency_counts.write_parquet("processed-data/7cii_high_frequency_counts.parquet")

high_frequency_counts.with_columns(pl.col(["share_of_sample", "share_of_visitors"]).round(5))

period,device_periods_in_sample,device_periods_visiting,flagged,share_of_sample,share_of_visitors
str,u32,u32,i64,f64,f64
"""201901""",23552,1029,4,0.00017,0.00389
"""201902""",23552,979,3,0.00013,0.00306
"""201903""",23552,1003,4,0.00017,0.00399
"""201904""",23552,1097,5,0.00021,0.00456
"""201905""",23552,1159,6,0.00025,0.00518
…,…,…,…,…,…
"""202112""",26186,1907,6,0.00023,0.00315
"""2019""",282624,13121,39,0.00014,0.00297
"""2020""",458688,21502,37,0.00008,0.00172


In [15]:
# (1) Chart: high-frequency devices month by month
plot_df = (high_frequency_counts
    .filter(pl.col("period").str.len_chars() == 6)
    .with_columns((pl.col("period").str.slice(0, 4) + "-" + pl.col("period").str.slice(4, 2)).alias("month"))
)

fig2 = px.bar(
    plot_df.to_pandas(),
    x="month",
    y="flagged",
    labels={
        "month": "Month",
        "flagged": "Devices with more than 31 dispensary visits",
    },
    title="High-Frequency Devices by Month, Yearly Sample",
)

show(fig2, "high_frequency_by_month")

In [16]:
### PART D
## Output: [7d] one row per [2ai] sample device and quarter (every device, four quarters each, visits or not), with the flags carried forward (1/0)
##         Columns are "year", "quarter", "caid_hash", "retail_worker" (k = 75, that quarter), "delivery_worker" (that quarter), "high_frequency" (in at
##         least one month of the quarter; the flag itself is monthly, see [7ci]), "high_frequency_months", "any_flag"
## Name: 7d_device_flags_quarterly.parquet
## Output: [7d'] counts and shares of flagged device-quarters, by quarter, year (added up), and all years, for each flag, any flag, and each combination
##         Columns are "period", "flag", "device_quarters_in_sample", "device_quarters_visiting", "flagged", "share_of_sample", "share_of_visitors"
## Name: 7d_flag_counts.parquet
FLAGS = ["retail_worker", "delivery_worker", "high_frequency"]
QUARTER_KEY = ["year", "quarter", "caid_hash"]

high_frequency_quarters = (high_frequency
    .with_columns((pl.col("year").cast(pl.String) + "Q" + ((pl.col("month") % 100 - 1) // 3 + 1).cast(pl.String)).alias("quarter"))
    .group_by(QUARTER_KEY)
    .agg(pl.col("high_frequency").max(), pl.col("high_frequency").sum().alias("high_frequency_months"))
)
device_flags = (windows.select(["year", "caid_hash"])
    .join(pl.DataFrame({"q": [1, 2, 3, 4]}), how = "cross")
    .with_columns((pl.col("year").cast(pl.String) + "Q" + pl.col("q").cast(pl.String)).alias("quarter"))
    .join(retail.group_by(QUARTER_KEY).agg(pl.col(f"retail_k{K_MAIN}").max().alias("retail_worker")), on = QUARTER_KEY, how = "left")
    .join(delivery.select(QUARTER_KEY + ["delivery_worker"]), on = QUARTER_KEY, how = "left")
    .join(high_frequency_quarters, on = QUARTER_KEY, how = "left")
    .with_columns(pl.col(FLAGS + ["high_frequency_months"]).fill_null(0).cast(pl.Int8))
    .with_columns(pl.max_horizontal(FLAGS).alias("any_flag"))
    .select(QUARTER_KEY + FLAGS + ["high_frequency_months", "any_flag"])
    .sort(["quarter", "caid_hash"])
)
device_flags.write_parquet("processed-data/7d_device_flags_quarterly.parquet")

# (1) Counts: each flag, any flag, and the overlaps (in the same quarter)
visitors = visits.select(QUARTER_KEY).unique().with_columns(pl.lit(1).alias("visiting"))
counted = device_flags.join(visitors, on = QUARTER_KEY, how = "left").with_columns(pl.col("visiting").fill_null(0))
FLAG_SETS = [(f, pl.col(f) == 1) for f in FLAGS + ["any_flag"]] + [
    ("retail and delivery", (pl.col("retail_worker") == 1) & (pl.col("delivery_worker") == 1)),
    ("retail and high frequency", (pl.col("retail_worker") == 1) & (pl.col("high_frequency") == 1)),
    ("delivery and high frequency", (pl.col("delivery_worker") == 1) & (pl.col("high_frequency") == 1)),
    ("all three", (pl.col("retail_worker") == 1) & (pl.col("delivery_worker") == 1) & (pl.col("high_frequency") == 1)),
]
PERIODS = ([(q, counted.filter(pl.col("quarter") == q)) for q in sorted(counted["quarter"].unique())]
           + [(YEAR, counted.filter(pl.col("year") == int(YEAR))) for YEAR in YEARS] + [("all", counted)])
results = []
for label, rows in PERIODS:
    for flag, condition in FLAG_SETS:
        results.append(rows.select(
            pl.lit(label).alias("period"),
            pl.lit(flag).alias("flag"),
            pl.len().alias("device_quarters_in_sample"),
            pl.col("visiting").sum().alias("device_quarters_visiting"),
            condition.sum().alias("flagged"),
        ))
flag_counts_table = (pl.concat(results)
    .with_columns((pl.col("flagged") / pl.col("device_quarters_in_sample")).alias("share_of_sample"),
                  (pl.col("flagged") / pl.col("device_quarters_visiting")).alias("share_of_visitors"))
)
flag_counts_table.write_parquet("processed-data/7d_flag_counts.parquet")

(flag_counts_table
    .filter(pl.col("period").is_in(YEARS + ["all"]))
    .with_columns((pl.col("share_of_sample") * 100).round(3).alias("pct_of_sample"), (pl.col("share_of_visitors") * 100).round(3).alias("pct_of_visitors")))

period,flag,device_quarters_in_sample,device_quarters_visiting,flagged,share_of_sample,share_of_visitors,pct_of_sample,pct_of_visitors
str,str,u32,i32,u32,f64,f64,f64,f64
"""2019""","""retail_worker""",94208,9012,1,0.000011,0.000111,0.001,0.011
"""2019""","""delivery_worker""",94208,9012,0,0.0,0.0,0.0,0.0
"""2019""","""high_frequency""",94208,9012,16,0.00017,0.001775,0.017,0.178
"""2019""","""any_flag""",94208,9012,17,0.00018,0.001886,0.018,0.189
"""2019""","""retail and delivery""",94208,9012,0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…
"""all""","""any_flag""",351848,37869,74,0.00021,0.001954,0.021,0.195
"""all""","""retail and delivery""",351848,37869,0,0.0,0.0,0.0,0.0
"""all""","""retail and high frequency""",351848,37869,1,0.000003,0.000026,0.0,0.003


In [17]:
# (2) Each flag quarter by quarter
(flag_counts_table
    .filter(pl.col("period").str.contains("Q") & pl.col("flag").is_in(FLAGS + ["any_flag"]))
    .pivot(on = "flag", index = "period", values = "flagged", maintain_order = True))

period,retail_worker,delivery_worker,high_frequency,any_flag
str,u32,u32,u32,u32
"""2019Q1""",0,0,4,4
"""2019Q2""",0,0,6,6
"""2019Q3""",1,0,3,4
"""2019Q4""",0,0,3,3
"""2020Q1""",2,0,7,8
…,…,…,…,…
"""2020Q4""",0,0,4,4
"""2021Q1""",0,0,8,8
"""2021Q2""",0,0,5,5


In [18]:
### PART E, HELPER
## The exclusions compared: nothing dropped, then each flag on its own, then any flag. Only the visits in a flagged period go: the device's visits in a
## quarter it was flagged as a retail or delivery worker, or in a month it was flagged high frequency. The device stays in the sample (and in the
## denominators) with the rest of its visits
EXCLUSIONS = [("none", None)] + [(f"excluding {f.replace('_', ' ')}", f) for f in FLAGS + ["any_flag"]]
FLAGGED = {
    "retail_worker": [(QUARTER_KEY, device_flags.filter(pl.col("retail_worker") == 1).select(QUARTER_KEY))],
    "delivery_worker": [(QUARTER_KEY, device_flags.filter(pl.col("delivery_worker") == 1).select(QUARTER_KEY))],
    "high_frequency": [(["month", "caid_hash"], high_frequency.filter(pl.col("high_frequency") == 1).select(["month", "caid_hash"]))],
}
FLAGGED["any_flag"] = FLAGGED["retail_worker"] + FLAGGED["delivery_worker"] + FLAGGED["high_frequency"]

def kept(visit_rows, flag): # the visits left after dropping the flagged periods
    for key, flagged in ([] if flag is None else FLAGGED[flag]):
        visit_rows = visit_rows.join(flagged, on = key, how = "anti")
    return visit_rows

def year_groups(df): # (label, rows) for each year, and all years pooled
    return [(YEAR, df.filter(pl.col("year") == int(YEAR))) for YEAR in YEARS] + [("all", df)]

QUANTILES = [(0.5, "median"), (0.75, "p75"), (0.90, "p90"), (0.95, "p95"), (0.99, "p99")]

In [19]:
### PART E, SUBPART I: FREQUENCY
## Output: [7ei] one row per exclusion, period (each year, "all"), and measure, like [4cii]: how many device-years visit a dispensary and, among those that
##         do, the distribution of the measure
##         Measures are "visits", "visits per 30 observed days", "stores (licenses)" (counting a store by its lowest operating license that month), and
##         "storefronts"
##         Columns are "exclusion", "period", "measure", "device_years" (every sample device, since only flagged periods' visits are dropped), "with_visit"
##         (device-years with a visit left), "share_with_visit", "total" (of the measure, over every device), then over the device-years with a visit:
##         "mean", "sd", "median", "p75", "p90", "p95", "p99", "max"
##         "visits per 30 observed days" keeps the whole year's observed days, even when a flagged quarter's visits were dropped
## Name: 7ei_frequency_by_exclusion.parquet
MEASURES = ["visits", "visits per 30 observed days", "stores (licenses)", "storefronts"]

def device_totals(visit_rows): # every sample device-year, with its visits (zeros included)
    return (windows
        .join(visit_rows
                .group_by(["year", "caid_hash"])
                .agg(
                    pl.len().alias("visits"),
                    pl.col("operating_licenses").list.min().n_unique().alias("stores (licenses)"),
                    pl.struct(STOREFRONT).n_unique().alias("storefronts"),
                ),
              on = ["year", "caid_hash"], how = "left")
        .with_columns(pl.col(["visits", "stores (licenses)", "storefronts"]).fill_null(0))
        .with_columns((pl.col("visits") / pl.col("observed_days") * 30).alias("visits per 30 observed days"))
    )

results = []
for exclusion, flag in EXCLUSIONS:
    for label, rows in year_groups(device_totals(kept(visits, flag))):
        visitors = rows.filter(pl.col("visits") > 0)
        for measure in MEASURES:
            results.append(visitors.select(
                pl.lit(exclusion).alias("exclusion"),
                pl.lit(label).alias("period"),
                pl.lit(measure).alias("measure"),
                pl.lit(rows.shape[0]).alias("device_years"),
                pl.len().alias("with_visit"),
                (pl.len() / rows.shape[0]).alias("share_with_visit"),
                pl.col(measure).cast(pl.Float64).sum().alias("total"),
                pl.col(measure).cast(pl.Float64).mean().alias("mean"),
                pl.col(measure).cast(pl.Float64).std().alias("sd"),
                *[pl.col(measure).cast(pl.Float64).quantile(q).alias(name) for q, name in QUANTILES],
                pl.col(measure).cast(pl.Float64).max().alias("max"),
            ))

frequency = pl.concat(results, how = "vertical_relaxed")
frequency.write_parquet("processed-data/7ei_frequency_by_exclusion.parquet")

frequency.filter(pl.col("measure") == "visits").with_columns(pl.col(["share_with_visit", "mean", "sd"]).round(3))

exclusion,period,measure,device_years,with_visit,share_with_visit,total,mean,sd,median,p75,p90,p95,p99,max
str,str,str,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""none""","""2019""","""visits""",23552,5110,0.217,25836.0,5.056,16.495,2.0,4.0,10.0,18.0,53.0,464.0
"""none""","""2020""","""visits""",38224,8117,0.212,42178.0,5.196,14.944,2.0,4.0,11.0,19.0,51.0,477.0
"""none""","""2021""","""visits""",26186,7381,0.282,48259.0,6.538,17.945,2.0,6.0,14.0,24.0,62.0,591.0
"""none""","""all""","""visits""",87962,20608,0.234,116273.0,5.642,16.468,2.0,5.0,12.0,21.0,55.0,591.0
"""excluding retail worker""","""2019""","""visits""",23552,5110,0.217,25770.0,5.043,16.388,2.0,4.0,10.0,18.0,53.0,464.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""excluding high frequency""","""all""","""visits""",87962,20608,0.234,110591.0,5.366,12.218,2.0,5.0,12.0,21.0,54.0,290.0
"""excluding any flag""","""2019""","""visits""",23552,5110,0.217,24178.0,4.732,11.322,2.0,4.0,10.0,18.0,52.0,212.0
"""excluding any flag""","""2020""","""visits""",38224,8117,0.212,40499.0,4.989,11.519,2.0,4.0,11.0,19.0,50.0,262.0


In [20]:
# (1) Visits per 30 observed days and stores, all years pooled
frequency.filter((pl.col("period") == "all") & (pl.col("measure") != "visits")).with_columns(pl.col(["share_with_visit", "mean", "sd"]).round(3))

exclusion,period,measure,device_years,with_visit,share_with_visit,total,mean,sd,median,p75,p90,p95,p99,max
str,str,str,i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""none""","""all""","""visits per 30 observed days""",87962,20608,0.234,9722.473081,0.472,1.375,0.166205,0.410959,1.002786,1.726027,4.60274,50.226629
"""none""","""all""","""stores (licenses)""",87962,20608,0.234,31411.0,1.524,1.704,1.0,2.0,3.0,3.0,6.0,137.0
"""none""","""all""","""storefronts""",87962,20608,0.234,31395.0,1.523,1.702,1.0,2.0,3.0,3.0,6.0,137.0
"""excluding retail worker""","""all""","""visits per 30 observed days""",87962,20608,0.234,9689.068821,0.47,1.359,0.166205,0.410959,1.002786,1.726027,4.576271,50.226629
"""excluding retail worker""","""all""","""stores (licenses)""",87962,20608,0.234,31410.0,1.524,1.704,1.0,2.0,3.0,3.0,6.0,137.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""excluding high frequency""","""all""","""stores (licenses)""",87962,20608,0.234,31267.0,1.517,1.355,1.0,2.0,3.0,3.0,6.0,86.0
"""excluding high frequency""","""all""","""storefronts""",87962,20608,0.234,31251.0,1.516,1.353,1.0,2.0,3.0,3.0,6.0,86.0
"""excluding any flag""","""all""","""visits per 30 observed days""",87962,20608,0.234,9216.958221,0.447,1.0,0.166205,0.410959,1.002786,1.726027,4.475138,22.266289


In [21]:
### PART E, SUBPART II: TIME
## Output: [7eii] one row per exclusion and period (each year, "all"), with when visits start and how long they last
##         Columns are "exclusion", "period", "visits", "share_weekend" (Saturday/Sunday starts), "share_outside_8am_midnight" (start hour 0-7),
##         "share_utc_midnight_start" (starts that may have been cut, see [4bi]), "mean_start_hour", then the minimum dwell (minutes): "dwell_mean",
##         "dwell_median", "dwell_p90", "dwell_p99", "share_hour_plus" (60+ minutes), "share_3h_plus"
## Name: 7eii_time_by_exclusion.parquet
## Output: [7eii'] start hours: one row per exclusion and hour (0-23), all years pooled, with "visits" and "share" (of the exclusion's visits)
## Name: 7eii_start_hours_by_exclusion.parquet
time_rows = []
hour_rows = []
for exclusion, flag in EXCLUSIONS:
    kept_visits = kept(visits, flag)
    for label, rows in year_groups(kept_visits):
        time_rows.append(rows.select(
            pl.lit(exclusion).alias("exclusion"),
            pl.lit(label).alias("period"),
            pl.len().alias("visits"),
            (pl.col("day_of_week") >= 6).mean().alias("share_weekend"),
            (pl.col("hour") < 8).mean().alias("share_outside_8am_midnight"),
            pl.col("utc_midnight_start").mean().alias("share_utc_midnight_start"),
            (pl.col("start_second") / 3600).mean().alias("mean_start_hour"),
            pl.col("minimum_dwell").mean().alias("dwell_mean"),
            pl.col("minimum_dwell").median().alias("dwell_median"),
            pl.col("minimum_dwell").quantile(0.90).alias("dwell_p90"),
            pl.col("minimum_dwell").quantile(0.99).alias("dwell_p99"),
            pl.col("hour_plus").mean().alias("share_hour_plus"),
            (pl.col("minimum_dwell") >= 180).mean().alias("share_3h_plus"),
        ))
    hour_rows.append(pl.DataFrame({"hour": range(24)}, schema = {"hour": pl.Int8})
        .join(kept_visits.group_by("hour").len().rename({"len": "visits"}), on = "hour", how = "left")
        .with_columns(pl.col("visits").fill_null(0))
        .with_columns((pl.col("visits") / pl.col("visits").sum()).alias("share"), pl.lit(exclusion).alias("exclusion"))
        .select(["exclusion", "hour", "visits", "share"]))

time_table = pl.concat(time_rows)
time_table.write_parquet("processed-data/7eii_time_by_exclusion.parquet")
start_hours = pl.concat(hour_rows)
start_hours.write_parquet("processed-data/7eii_start_hours_by_exclusion.parquet")

time_table.with_columns(pl.exclude(["exclusion", "period", "visits"]).round(4))

exclusion,period,visits,share_weekend,share_outside_8am_midnight,share_utc_midnight_start,mean_start_hour,dwell_mean,dwell_median,dwell_p90,dwell_p99,share_hour_plus,share_3h_plus
str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""none""","""2019""",25836,0.2642,0.0694,0.0455,14.5108,46.082,9.0,122.0,537.0,0.1584,0.0766
"""none""","""2020""",42178,0.2637,0.0744,0.0407,14.3823,49.2772,9.0,131.0,537.0,0.139,0.0861
"""none""","""2021""",48259,0.2704,0.0546,0.0499,14.7977,46.8141,9.0,122.0,531.0,0.1288,0.0839
"""none""","""all""",116273,0.2666,0.0651,0.0456,14.5833,47.5449,9.0,124.0,536.0,0.1391,0.0831
"""excluding retail worker""","""2019""",25770,0.2648,0.0673,0.0456,14.5309,45.0175,9.0,119.0,528.6667,0.1563,0.0745
…,…,…,…,…,…,…,…,…,…,…,…,…
"""excluding high frequency""","""all""",110591,0.2718,0.0544,0.0468,14.7006,43.876,9.0,95.0,535.0,0.1213,0.075
"""excluding any flag""","""2019""",24178,0.2747,0.0541,0.0472,14.6913,43.0186,9.0,114.0,527.0,0.1455,0.0731
"""excluding any flag""","""2020""",40499,0.2688,0.0575,0.0419,14.5215,43.4942,9.0,89.0,531.0,0.1175,0.0736


In [22]:
# (1) Chart: start hours before and after excluding every flagged quarter/month
fig3 = px.line(
    start_hours.filter(pl.col("exclusion").is_in(["none", "excluding any flag"])).with_columns(pl.col("share") * 100).to_pandas(),
    x="hour",
    y="share",
    color="exclusion",
    markers=True,
    labels={
        "hour": "Start hour (local)",
        "share": "Share of visits (%)",
        "exclusion": "",
    },
    title="Dispensary Visit Start Times, Before and After Excluding Flagged Quarters and Months, Yearly Sample",
)

show(fig3, "start_hours_by_exclusion")

In [23]:
### PART E, SUBPART III: REGION
## Output: [7eiii] one row per exclusion, period (each year, "all"), and level, like [5a]: the share of dispensary visits to a store in the device's home
##         county, city, ZIP, or CBG. Only devices with a home in [4ai] count, and at the city level only devices living in a city (place)
##         Columns are "exclusion", "period", "level", "devices", "visits", "visits_in_home_area", "share_of_visits" (pooled), "device_average_share"
## Name: 7eiii_home_area_by_exclusion.parquet
## Output: [7eiii'] one row per exclusion and store county (all years pooled), with "visits", "share" (of the exclusion's visits), and "devices" (device-years
##         visiting a store there)
## Name: 7eiii_store_counties_by_exclusion.parquet
LEVELS = [
    ("county", "home_county_fips", "store_county_fips"),
    ("city", "home_place_fips", "store_place_fips"),
    ("zip", "home_zip_code", "store_zcta"), # the device's (USPS) home zip against the ZCTA the store sits in, as in [5a]
    ("cbg", "home_cbg", "store_cbg"),
]

area_rows = []
county_rows = []
for exclusion, flag in EXCLUSIONS:
    kept_visits = kept(visits, flag)
    with_homes = kept_visits.join(crosswalk, on = ["year", "caid_hash"], how = "inner")
    for LEVEL, HOME, STORE in LEVELS:
        level_visits = (with_homes
            .filter(pl.col(HOME).is_not_null())
            .with_columns((pl.col(HOME) == pl.col(STORE)).fill_null(False).alias("in_home_area"))
        )
        for label, rows in year_groups(level_visits):
            device_shares = rows.group_by(["year", "caid_hash"]).agg(pl.len().alias("visits"), pl.col("in_home_area").sum().alias("visits_in_home_area"))
            area_rows.append(device_shares.select(
                pl.lit(exclusion).alias("exclusion"),
                pl.lit(label).alias("period"),
                pl.lit(LEVEL).alias("level"),
                pl.len().alias("devices"),
                pl.col("visits").sum(),
                pl.col("visits_in_home_area").sum(),
                (pl.col("visits_in_home_area").sum() / pl.col("visits").sum()).alias("share_of_visits"),
                (pl.col("visits_in_home_area") / pl.col("visits")).mean().alias("device_average_share"),
            ))
    county_rows.append(kept_visits
        .group_by("store_county_name")
        .agg(pl.len().alias("visits"), pl.struct(["year", "caid_hash"]).n_unique().alias("devices"))
        .with_columns((pl.col("visits") / pl.col("visits").sum()).alias("share"), pl.lit(exclusion).alias("exclusion"))
        .select(["exclusion", "store_county_name", "visits", "share", "devices"])
        .sort("visits", descending = True))

home_areas = pl.concat(area_rows, how = "vertical_relaxed")
home_areas.write_parquet("processed-data/7eiii_home_area_by_exclusion.parquet")
store_counties = pl.concat(county_rows)
store_counties.write_parquet("processed-data/7eiii_store_counties_by_exclusion.parquet")

(home_areas
    .filter(pl.col("period") == "all")
    .pivot(on = "level", index = "exclusion", values = "share_of_visits")
    .with_columns(pl.col(["county", "city", "zip", "cbg"]).round(4)))

exclusion,county,city,zip,cbg
str,f64,f64,f64,f64
"""none""",0.868,0.3633,0.3159,0.0384
"""excluding retail worker""",0.8676,0.3639,0.3152,0.0386
"""excluding delivery worker""",0.8682,0.3638,0.3163,0.0385
"""excluding high frequency""",0.8665,0.3566,0.3,0.0393
"""excluding any flag""",0.8661,0.357,0.2993,0.0394


In [24]:
# (1) Store counties, shares of visits (in %) before and after each exclusion
(store_counties
    .with_columns((pl.col("share") * 100).round(2))
    .pivot(on = "exclusion", index = "store_county_name", values = "share")
    .sort("none", descending = True))

store_county_name,none,excluding retail worker,excluding delivery worker,excluding high frequency,excluding any flag
str,f64,f64,f64,f64,f64
"""King""",14.31,14.35,14.32,14.49,14.54
"""Spokane""",12.21,12.25,12.23,12.18,12.22
"""Pierce""",11.6,11.58,11.61,12.01,12.0
"""Snohomish""",10.36,10.4,10.37,10.71,10.75
"""Thurston""",7.83,7.85,7.84,7.98,8.01
…,…,…,…,…,…
"""Okanogan""",0.07,0.07,0.07,0.07,0.07
"""Columbia""",0.07,0.07,0.07,0.07,0.07
"""Lincoln""",0.06,0.06,0.06,0.06,0.06


In [25]:
# (2) Chart: how much each exclusion changes the visit totals and the share of devices visiting, by year
plot_df = (frequency
    .filter((pl.col("measure") == "visits") & (pl.col("period") != "all"))
    .with_columns((pl.col("share_with_visit") * 100).alias("share_pct"))
)

fig4 = px.bar(
    plot_df.to_pandas(),
    x="period",
    y="mean",
    color="exclusion",
    barmode="group",
    labels={
        "period": "Year",
        "mean": "Mean dispensary visits per visiting device",
        "exclusion": "",
    },
    title="Visits per Visiting Device, Before and After Exclusions, Yearly Sample",
)

show(fig4, "visits_per_visitor_by_exclusion")

In [26]:
### CHECKS
# (1) The flag tables add up: [7d] has four quarters per sample device-year, and its flags are exactly the flagged quarters of [7ai]/[7bi] and the
#     quarters holding a flagged month of [7ci]
print("[7d] rows = 4 x sample device-years:", device_flags.shape[0] == 4 * windows.shape[0])
print("retail flags match [7ai]:", device_flags["retail_worker"].sum() == retail.filter(pl.col(f"retail_k{K_MAIN}") == 1).select(QUARTER_KEY).unique().shape[0])
print("delivery flags match [7bi]:", device_flags["delivery_worker"].sum() == delivery["delivery_worker"].sum())
print("high-frequency months match [7ci]:", device_flags["high_frequency_months"].cast(pl.Int64).sum() == high_frequency["high_frequency"].cast(pl.Int64).sum())
print("visits dropped by 'any flag' = visits in flagged quarters/months:", visits.shape[0] - kept(visits, "any_flag").shape[0])

# (2) A higher k can only flag fewer devices
print("flags shrink as k grows:", all((retail[f"retail_k{a}"] >= retail[f"retail_k{b}"]).all() for a, b in zip(K_VALUES, K_VALUES[1:])))

# (3) "none" in [7ei] is the same as [4cii] for the yearly sample
check = (pl.read_parquet("processed-data/4cii_visit_distributions.parquet")
    .filter((pl.col("sample") == "yearly") & (pl.col("measure") == "visits"))
    .join(frequency.filter((pl.col("exclusion") == "none") & (pl.col("measure") == "visits")), on = "period", how = "inner"))
print("[7ei] 'none' matches [4cii]:", check.shape[0], "periods,", (check["with_visit"] == check["with_visit_right"]).all() and ((check["mean"] - check["mean_right"]).abs() < 1e-9).all())

[7d] rows = 4 x sample device-years: True
retail flags match [7ai]: True
delivery flags match [7bi]: True
high-frequency months match [7ci]: True
visits dropped by 'any flag' = visits in flagged quarters/months: 6053
flags shrink as k grows: True
[7ei] 'none' matches [4cii]: 4 periods, True


### NOTES
- Everything here is counted in device-quarters (device-months for high frequency): a device is a worker in the quarters it qualifies in, not for the whole year. The sample has 351,848 device-quarters (87,962 device-years x 4), 37,869 of them with a dispensary visit.
- Retail workers: 52 device-quarters have 4+ weekdays at one license in 9+ weeks; 31 of those also have 75%+ of their visits there lasting an hour or more. The regular-times test then flags 30 (k = 25), 29 (33), 18 (50), 8 (66), 6 (75), and 1 (90). At k = 75 that's 0.002% of sample device-quarters and 0.016% of the ones with a dispensary visit. Most candidates that fail it have two or three clusters of start times (different shifts), not a spread.
- Looser retail rules (k = 75): 8+ weeks flags 8 device-quarters, 6+ weeks 12, and 3+ weekdays in 9+ weeks 14 (against 6). At k = 25 they flag 42, 58, and 58. The weekly pattern itself is rare (52 to 94 device-quarters pass it depending on the rule), and about a third of those fail the dwell test.
- With the best hour instead of the best half hour, the main rule flags 31, 31, 22, 11, 8, and 1 device-quarters for k = 25 to 90 (against 30, 29, 18, 8, 6, 1), and the loosest rules reach 19-20 at k = 75. Doubling the window adds little because the candidates that fail mostly have start times in separate clusters hours apart (different shifts), which no single hour covers.
- Delivery/wholesale workers: 14 device-quarters have 9+ weeks with 3+ visits to 2+ storefronts, but only 1 has 75%+ of its visits lasting an hour or more. Most of the rest stay a few minutes per stop, which is what a delivery looks like, so the dwell requirement is what keeps this flag near zero. Looser: 8+ weeks still flags 1 (19 pass the weekly pattern), 6+ weeks flags 4 (37).
- High frequency: 133 device-months (in 69 device-quarters), 57 of them in 2021. Several are the same devices month after month, at one storefront, with short visits (eg. 142 visits in Oct 2021 on 31 distinct days, 16% an hour or longer): these look like employees whose visits Veraset splits up, and they fail the retail dwell test for exactly that reason.
- Any flag: 74 device-quarters (0.02% of sample device-quarters, 0.20% of the ones with a visit), holding 6,053 visits (5.2% of all dispensary visits). Overlaps in the same quarter: 1 retail + high frequency, 1 delivery + high frequency.
- Dropping the flagged quarters and months leaves every device in the sample and doesn't change the share of devices with a visit (23.43%) or the median (2 a year), but trims the tail: mean visits per visitor 5.64 to 5.35, p99 55 to 53, max 591 to 262. Visits starting outside 8 am-midnight fall from 6.5% to 5.2%, visits of an hour or more from 13.9% to 11.9%, and the 90th percentile dwell from 124 to 89 minutes. Home ZIP shares fall from 31.6% to 29.9% and home city from 36.3% to 35.7%, since the flagged devices mostly go to one store near home. Most of this comes from the high-frequency flag.
- "Visits per 30 observed days" in [7ei] keeps the whole year's observed days after a flagged quarter's visits are dropped, so it understates those devices' rate a little.
- The flags are strict: devices with 100+ visits a year that don't meet every criterion in some quarter stay in. The tables in Parts A-C keep every criterion, so looser versions (a lower dwell share, other week counts) can be read off them without rerunning.
